# Module 06 — Behavioral Cloning

Imitation learning — the dominant paradigm in modern robotics. See [the lesson](README.md).

> Tip: in Colab, set **Runtime → Change runtime type → GPU**.

In [ ]:
# === Colab setup: run me first ===
import os, sys
if not os.path.exists('rl'):
    # On Colab, clone the repo so the `rl` package is importable.
    !git clone https://github.com/anhduckkzz/lunarlander.git repo && (cp -r repo/* . 2>/dev/null || true)
    !pip -q install 'gymnasium[box2d]>=0.29' torch numpy matplotlib imageio tqdm
import torch
print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Clone an expert PPO/DQN policy with supervised learning
First train (or load) an expert, then clone it and compare returns.

In [ ]:
from rl.envs import make_env, env_dims
from rl.agents.dqn import DQN, DQNConfig
from rl.agents.bc import BehavioralCloning, BCConfig, collect_demonstrations
from rl.train import train_offpolicy
from rl.utils import ExponentialSchedule, set_seed
import numpy as np

set_seed(0)
env = make_env('LunarLander-v3', seed=0)
s_dim, a_dim, _ = env_dims(env)

# 1) Train a quick expert (use more steps for a stronger teacher)
expert = DQN(s_dim, a_dim, DQNConfig(), device=DEVICE)
train_offpolicy(expert, env, n_steps=120_000,
                eps_schedule=ExponentialSchedule(1.0, 0.01, 0.999), solved_at=200)

# 2) Collect demonstrations and clone them (pure supervised learning)
states, actions = collect_demonstrations(env, expert, n_episodes=50)
bc = BehavioralCloning(s_dim, a_dim, discrete=True, cfg=BCConfig(epochs=80), device=DEVICE)
bc.fit(states, actions)

### Evaluate the clone — and witness covariate shift

In [ ]:
import numpy as np
def eval_agent(agent, n=20, greedy_dqn=False):
    rets = []
    for _ in range(n):
        s, _ = env.reset(); done=False; R=0
        while not done:
            a = agent.act(s, eps=0.0) if greedy_dqn else agent.act(s)
            s, r, t, tr, _ = env.step(a); R += r; done = t or tr
        rets.append(R)
    return np.mean(rets)
print('expert return :', round(eval_agent(expert, greedy_dqn=True), 1))
print('clone  return :', round(eval_agent(bc), 1))
print('\nIf the clone underperforms the expert, that gap is covariate shift —',
      'states the expert never visited. Fixes: more/recovery data, DAgger, RL fine-tuning.')